# 01 — Download ACS Data from MAST

Downloads all ACS/WFC FLC and FLT files for proposal 14706 from MAST and organizes
them into `data_acs/raw/<target_name>/`.

FLC files have the pixel-based CTE correction already applied by CalACS (Anderson & Bedin 2010).
FLT-only files require a separate CTE correction step before they can be drizzled.

All parameters read from `config/wfc3_ir_drizzle_params.yaml` — nothing hardcoded here.

In [ ]:
import shutil
import yaml
from collections import defaultdict
from pathlib import Path
from astropy.io import fits
from astropy.table import Table
from astroquery.mast import Observations

with open('../config/wfc3_ir_drizzle_params.yaml') as f:
    cfg = yaml.safe_load(f)

dl_cfg = cfg['acs_download']
RAW_DIR = Path('../data_acs/raw')

In [ ]:
# Query MAST for all ACS/WFC observations from proposal 14706
# project='HST' is required — 'HAP' returns processed mosaics, not individual exposures
obs_table = Observations.query_criteria(
    proposal_id=dl_cfg['proposal_id'],
    instrument_name=dl_cfg['instrument_name'],
    project=dl_cfg['project'],
)
print(f'Observations found: {len(obs_table)}')

# Retrieve the full product list, then filter to FLC and FLT science files only
products = Observations.get_product_list(obs_table)
filtered = Observations.filter_products(
    products,
    productSubGroupDescription=dl_cfg['product_sub_group'],
    productType='SCIENCE',
)
print(f'Products to download: {len(filtered)}')

# cache=True skips any files already present in mastDownload/ on disk
Observations.download_products(filtered, cache=True)

In [ ]:
# Move each file from mastDownload/HST/ into data_acs/raw/<TARGNAME>/
# Reads TARGNAME from the FITS primary header — more reliable than MAST metadata
# Skips files already in place so this cell is safe to re-run
# HAP-pipeline files (hst_<proposal>_... naming) are skipped — they are duplicates
# of the standard CalACS files and are not used in this pipeline

moved, skipped, skipped_hap = 0, 0, 0
mast_dir = Path('./mastDownload/HST')

if mast_dir.exists():
    for f in sorted(mast_dir.glob('*/*fl?.fits')):
        if f.name.startswith('hst_'):
            skipped_hap += 1
            continue
        target = fits.getval(str(f), 'TARGNAME', ext=0)
        dest_dir = RAW_DIR / target
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / f.name
        if dest.exists():
            skipped += 1
        else:
            shutil.move(str(f), str(dest))
            moved += 1

print(f'Moved {moved} files, skipped {skipped} already in place, skipped {skipped_hap} HAP duplicates')

In [ ]:
# Verify what is on disk against the expected file types from config
#
# For each target/filter pair, check whether FLC files are present.
# FLC present  → CTE correction already applied, ready for alignment
# FLT only     → CTE correction needed before alignment
#
# Status is OK when the presence of FLC matches what the config expects.

def get_filter(hdr):
    # ACS/WFC places science filters in either FILTER1 or FILTER2;
    # the other slot holds a CLEAR element. Return whichever is not CLEAR.
    f1, f2 = hdr.get('FILTER1', ''), hdr.get('FILTER2', '')
    return f1 if 'CLEAR' not in f1 else f2

counts = defaultdict(lambda: {'FLC': 0, 'FLT': 0})
for f in sorted(RAW_DIR.glob('*/*fl?.fits')):
    target = f.parent.name
    with fits.open(f) as hdul:
        filt = get_filter(hdul[0].header)
    suffix = 'FLC' if f.name.endswith('flc.fits') else 'FLT'
    counts[(target, filt)][suffix] += 1

expected = cfg['acs_targets']

rows = []
all_ok = True
for target in sorted(expected):
    for filt in sorted(expected[target]):
        exp_type = expected[target][filt]
        key = (target, filt)
        n_flc = counts[key]['FLC']
        n_flt = counts[key]['FLT']
        has_flc = n_flc > 0
        status = 'OK' if (has_flc == (exp_type == 'FLC')) else 'MISMATCH'
        if status != 'OK':
            all_ok = False
        rows.append((target, filt, n_flc, n_flt, exp_type, status))

t = Table(rows=rows, names=['Target', 'Filter', 'N_FLC', 'N_FLT', 'Expected', 'Status'])
t.pprint(max_lines=-1, max_width=110)
print()
print('All checks passed.' if all_ok else 'WARNING: mismatches found — check Expected vs actual FLC presence.')